

## Conceptos clave del texto
- **Individuo**: posee un **genotipo** (información genética) y un **fenotipo** (características observables).
- **Cromosoma**: secuencia de **genes** (unidades de información); el valor de un gen es su **alelo**.
- **Mutación**: alteración aleatoria en la cadena de ADN que puede resultar favorable o desfavorable.
- **Cruzamiento (crossover)**: intercambio de segmentos entre cromosomas mediante **quiasmas**.
- **Aptitud (fitness)**: qué tan bien adaptado está un individuo a su entorno.
- **Población**: conjunto de individuos que coexisten y evolucionan a lo largo de **generaciones**.
- **Selección natural**: los individuos más aptos tienen mayor probabilidad de reproducirse.

In [1]:
import random
import copy


## Clase `Individuo`
Representa un organismo dentro de la población.
Su **genotipo** es una cadena de bits (cromosoma binario).
Su **fenotipo** y **aptitud** se derivan de ese genotipo.

In [2]:
class Individuo:
    """
    Modela un individuo dentro de un Algoritmo Genético.

    Atributos
    ---------
    longitud_cromosoma : int
        Número de genes (bits) en el cromosoma.
    cromosoma : list[int]
        Genotipo binario del individuo [0, 1, 1, 0, ...].
    aptitud : float
        Valor de aptitud (fitness) calculado por la función objetivo.
    """

    def __init__(self, longitud_cromosoma: int, cromosoma: list = None):
        self.longitud_cromosoma = longitud_cromosoma
        # Si no se proporciona cromosoma, se genera aleatoriamente (individuo nuevo)
        self.cromosoma = cromosoma if cromosoma else self._generar_cromosoma()
        self.aptitud = 0.0

    # ------------------------------------------------------------------ #
    # Métodos privados                                                     #
    # ------------------------------------------------------------------ #

    def _generar_cromosoma(self) -> list:
        """Crea un cromosoma binario aleatorio (genotipo inicial)."""
        return [random.randint(0, 1) for _ in range(self.longitud_cromosoma)]

    # ------------------------------------------------------------------ #
    # Métodos públicos                                                     #
    # ------------------------------------------------------------------ #

    def calcular_aptitud(self, funcion_objetivo=None) -> float:
        """
        Evalúa qué tan bien adaptado está el individuo.
        Por defecto, suma los genes activos (problema OneMax).
        Se puede inyectar cualquier función objetivo externa.
        """
        if funcion_objetivo:
            self.aptitud = funcion_objetivo(self.cromosoma)
        else:
            # OneMax: contar cuántos genes tienen valor 1
            self.aptitud = sum(self.cromosoma)
        return self.aptitud

    def mutar(self, probabilidad_mutacion: float = 0.01) -> None:
        """
        Aplica mutación puntual al cromosoma.
        Cada gen puede invertirse con 'probabilidad_mutacion'.
        Simula el error de la ADN-polimerasa descrito en la sección 1.2.
        """
        for i in range(self.longitud_cromosoma):
            if random.random() < probabilidad_mutacion:
                self.cromosoma[i] = 1 - self.cromosoma[i]   # flip del alelo

    def fenotipo(self) -> int:
        """
        Decodifica el genotipo binario a un valor entero (fenotipo).
        Interpreta el cromosoma como número en base 2.
        """
        return int(''.join(map(str, self.cromosoma)), 2)

    def __repr__(self) -> str:
        genes = ''.join(map(str, self.cromosoma))
        return (f"Individuo(genotipo={genes}, "
                f"fenotipo={self.fenotipo()}, aptitud={self.aptitud:.2f})")

In [3]:
class Poblacion:
    """
    Modela la población de un Algoritmo Genético.

    Atributos
    ---------
    tamano : int
        Número de individuos en la población.
    longitud_cromosoma : int
        Longitud del cromosoma de cada individuo.
    individuos : list[Individuo]
        Colección de individuos actuales.
    generacion : int
        Contador de generaciones evolutivas.
    historial_aptitud : list[float]
        Mejor aptitud registrada por generación.
    """

    def __init__(self, tamano: int, longitud_cromosoma: int):
        self.tamano = tamano
        self.longitud_cromosoma = longitud_cromosoma
        self.generacion = 0
        self.historial_aptitud = []
        self.individuos = self._inicializar()

    # ------------------------------------------------------------------ #
    # Métodos privados                                                     #
    # ------------------------------------------------------------------ #

    def _inicializar(self) -> list:
        """Crea la generación inicial con individuos aleatorios."""
        return [Individuo(self.longitud_cromosoma) for _ in range(self.tamano)]

    # ------------------------------------------------------------------ #
    # Métodos públicos                                                     #
    # ------------------------------------------------------------------ #

    def evaluar(self, funcion_objetivo=None) -> None:
        """Calcula la aptitud de todos los individuos de la población."""
        for ind in self.individuos:
            ind.calcular_aptitud(funcion_objetivo)

    def seleccionar(self, k: int = 3) -> 'Individuo':
        """
        Selección por torneo: elige k individuos al azar y retorna
        el más apto. Simula la competencia descrita en la sección 1.1.
        """
        competidores = random.sample(self.individuos, k)
        return max(competidores, key=lambda ind: ind.aptitud)

    def cruzar(self, padre: 'Individuo', madre: 'Individuo') -> tuple:
        """
        Cruzamiento de un punto (one-point crossover).
        Simula el quiasma descrito en la sección 1.2:
        se elige un punto de corte y se intercambian segmentos.
        Retorna dos hijos (Individuo).
        """
        punto_corte = random.randint(1, self.longitud_cromosoma - 1)
        crom_hijo1 = padre.cromosoma[:punto_corte] + madre.cromosoma[punto_corte:]
        crom_hijo2 = madre.cromosoma[:punto_corte] + padre.cromosoma[punto_corte:]
        return Individuo(self.longitud_cromosoma, crom_hijo1), \
               Individuo(self.longitud_cromosoma, crom_hijo2)

    def evolucionar(self, prob_mutacion: float = 0.01,
                    funcion_objetivo=None) -> None:
        """
        Ejecuta un ciclo completo de evolución:
        1. Evaluar aptitud
        2. Seleccionar padres (torneo)
        3. Cruzar para generar hijos
        4. Mutar hijos
        5. Reemplazar población
        """
        self.evaluar(funcion_objetivo)
        nueva_poblacion = []

        while len(nueva_poblacion) < self.tamano:
            padre = self.seleccionar()
            madre = self.seleccionar()
            hijo1, hijo2 = self.cruzar(padre, madre)
            hijo1.mutar(prob_mutacion)
            hijo2.mutar(prob_mutacion)
            nueva_poblacion.extend([hijo1, hijo2])

        self.individuos = nueva_poblacion[:self.tamano]
        self.generacion += 1
        self.historial_aptitud.append(self.mejor_individuo().aptitud)

    def mejor_individuo(self) -> 'Individuo':
        """Retorna el individuo con mayor aptitud en la generación actual."""
        return max(self.individuos, key=lambda ind: ind.aptitud)

    def aptitud_promedio(self) -> float:
        """Calcula la aptitud promedio de la población."""
        return sum(ind.aptitud for ind in self.individuos) / self.tamano

    def __repr__(self) -> str:
        return (f"Poblacion(tamaño={self.tamano}, "
                f"generacion={self.generacion}, "
                f"mejor_aptitud={self.mejor_individuo().aptitud:.2f})")


## Clase `Poblacion`
Conjunto de individuos que evoluciona a lo largo de generaciones
mediante **selección**, **cruzamiento** y **mutación**.


## Demo — Ejecutando el AG (problema OneMax)

In [4]:
# Parámetros del AG
TAMANO_POBLACION   = 20
LONGITUD_CROMOSOMA = 10
GENERACIONES       = 30
PROB_MUTACION      = 0.02

# Crear población inicial
poblacion = Poblacion(TAMANO_POBLACION, LONGITUD_CROMOSOMA)
poblacion.evaluar()

print(f"{'Gen':>4} | {'Mejor aptitud':>14} | {'Aptitud promedio':>16} | Mejor individuo")
print("-" * 70)

for gen in range(GENERACIONES):
    poblacion.evolucionar(PROB_MUTACION)
    mejor = poblacion.mejor_individuo()
    mejor.calcular_aptitud()   # re-evaluar tras cruzamiento
    promedio = poblacion.aptitud_promedio()
    print(f"{gen+1:>4} | {mejor.aptitud:>14.1f} | {promedio:>16.2f} | {mejor}")

print("\n✅ Evolución completada.")
print(f"Mejor individuo final: {poblacion.mejor_individuo()}")

 Gen |  Mejor aptitud | Aptitud promedio | Mejor individuo
----------------------------------------------------------------------
   1 |           10.0 |             0.50 | Individuo(genotipo=1111111111, fenotipo=1023, aptitud=10.00)
   2 |            8.0 |             0.40 | Individuo(genotipo=1111111010, fenotipo=1018, aptitud=8.00)
   3 |            8.0 |             0.40 | Individuo(genotipo=1110101111, fenotipo=943, aptitud=8.00)
   4 |            9.0 |             0.45 | Individuo(genotipo=1011111111, fenotipo=767, aptitud=9.00)
   5 |           10.0 |             0.50 | Individuo(genotipo=1111111111, fenotipo=1023, aptitud=10.00)
   6 |           10.0 |             0.50 | Individuo(genotipo=1111111111, fenotipo=1023, aptitud=10.00)
   7 |            9.0 |             0.45 | Individuo(genotipo=1111111110, fenotipo=1022, aptitud=9.00)
   8 |           10.0 |             0.50 | Individuo(genotipo=1111111111, fenotipo=1023, aptitud=10.00)
   9 |           10.0 |             0.50 | I